# ELIQSIR — Step 2: Transformation & Loading
**Integração e Processamento Analítico de Informação · 2025/2026**

This notebook transforms the four raw extracted datasets produced in **Step 1** into a clean star-schema data warehouse.

Pipeline stages covered here:
1. Load raw Parquet files produced by Step 1
2. Clean & normalise each dimension source
3. Build `FactBioactivity`
4. Initialise the warehouse schema (from `src/database/schema.sql`)
5. Persist dimensions & fact as Parquet (staging area)
6. Load staging Parquet files into the warehouse
7. Validate loaded data

> **Pre-requisite:** Run `notebooks/01_extraction_demo.ipynb` first so that  
> the following Parquet files exist inside `data/raw/`:
> - `raw_uniprot.parquet`
> - `raw_chembl.parquet`
> - `raw_pdbe.parquet`
> - `raw_pubmed.parquet`


## 1 · Import Libraries & Configuration

We add the project root to `sys.path` so that the `src/` package can be imported without installing it.
All configuration (database URL, API credentials, file paths) is centralised in `src/config.py` and read from the `.env` file at the project root.


In [1]:
import sys
import re
import sqlite3
from pathlib import Path
from typing import Callable

# ── make src/ importable ──────────────────────────────────────────────────────
REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np

from src.config import settings
from src.transformation.cleaner import DataCleaner
from src.transformation.dimensional_builder import DimensionalModelBuilder

# Shorthand for safe display of paths (project-relative, never absolute)
_dp = settings.display_path

# ── directories ──────────────────────────────────────────────────────────────
DATA_DIR      = REPO_ROOT / "data"
RAW_DIR       = DATA_DIR  / "raw"           # Parquet files from Step 1
STAGING_DIR   = DATA_DIR  / "staging"       # Parquet files after cleaning/modelling
WAREHOUSE_DB  = DATA_DIR  / "eliqsir_warehouse.db"   # SQLite demo warehouse
SCHEMA_SQL    = REPO_ROOT / "src" / "database" / "schema.sql"

STAGING_DIR.mkdir(parents=True, exist_ok=True)

print("Raw Parquet dir :", _dp(RAW_DIR))
print("Staging dir     :", _dp(STAGING_DIR))
print("Warehouse DB    :", _dp(WAREHOUSE_DB))
print("Schema SQL      :", _dp(SCHEMA_SQL))
print("NCBI e-mail     :", settings.ncbi_email or "⚠  NOT SET — add NCBI_EMAIL to .env")


Raw Parquet dir : data/raw
Staging dir     : data/staging
Warehouse DB    : data/eliqsir_warehouse.db
Schema SQL      : src/database/schema.sql
NCBI e-mail     : pedro.fanica@gmail.com


## 2 · Load Raw Extracted Data

We read the four **Parquet** files written by `01_extraction_demo.ipynb`.  
Parquet preserves column dtypes (including nullable integers) and is ~10× faster to load than CSV for large datasets.

If a file is missing, an empty DataFrame is used so the rest of the notebook can still demonstrate the transformations.


In [2]:
def load_parquet(path: Path, label: str) -> pd.DataFrame:
    """Load a Parquet file with a friendly fallback when the file is absent."""
    if not path.exists():
        print(f"  ⚠  {label}: file not found at {_dp(path)}")
        print("     Run 01_extraction_demo.ipynb first to generate it.")
        return pd.DataFrame()
    df = pd.read_parquet(path)
    print(f"  ✓  {label}: {df.shape[0]:,} rows × {df.shape[1]} columns")
    return df


print("Loading raw Parquet files …")
df_raw_uniprot    = load_parquet(RAW_DIR / "raw_uniprot.parquet",  "UniProt proteins")
df_raw_chembl     = load_parquet(RAW_DIR / "raw_chembl.parquet",   "ChEMBL activities")
df_raw_pdbe       = load_parquet(RAW_DIR / "raw_pdbe.parquet",     "PDB structures")
df_raw_pubmed     = load_parquet(RAW_DIR / "raw_pubmed.parquet",   "PubMed abstracts")


Loading raw Parquet files …
  ✓  UniProt proteins: 20,431 rows × 8 columns
  ✓  ChEMBL activities: 6,441,095 rows × 23 columns
  ✓  PDB structures: 168,406 rows × 8 columns
  ✓  PubMed abstracts: 35,551 rows × 7 columns
  ✓  ChEMBL activities: 6,441,095 rows × 23 columns
  ✓  PDB structures: 168,406 rows × 8 columns
  ✓  PubMed abstracts: 35,551 rows × 7 columns


In [3]:
# Quick sanity preview
for label, df in [
    ("UniProt",    df_raw_uniprot),
    ("ChEMBL",     df_raw_chembl),
    ("PDBe",       df_raw_pdbe),
    ("PubMed",     df_raw_pubmed),
]:
    if not df.empty:
        print(f"\n{'─'*60}")
        print(f"  {label}  —  columns: {list(df.columns)}")
        display(df.head(3))



────────────────────────────────────────────────────────────
  UniProt  —  columns: ['accession', 'gene_names', 'protein_name', 'organism_name', 'protein_sequence', 'protein_class', 'ec_number', 'catalyzed_reaction']


,accession,gene_names,protein_name,organism_name,protein_sequence,protein_class,ec_number,catalyzed_reaction
0,A0A087X1C5,CYP2D7,Cytochrome P450 2D7 (EC 1.14.14.1),Homo sapiens (Human),MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,Cytochrome P450 family,1.14.14.1,CATALYTIC ACTIVITY: Reaction=an organic molecu...
1,A0A096LP01,SMIM26 LINC00493,Small integral membrane protein 26,Homo sapiens (Human),MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,SMIM26 family,None,None
2,A0A0B4J2F0,PIGBOS1,Protein PIGBOS1 (PIGB opposite strand protein 1),Homo sapiens (Human),MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,None,None,None



────────────────────────────────────────────────────────────
  ChEMBL  —  columns: ['activity_id', 'drug_chembl_id', 'drug_name', 'molecule_type', 'molecular_weight', 'canonical_smiles', 'target_chembl_id', 'target_name', 'organism', 'standard_type', 'standard_value', 'standard_units', 'pchembl_value', 'assay_type', 'assay_description', 'assay_organism', 'confidence_score', 'article_title', 'journal', 'year', 'pubmed_id', 'doi', 'uniprot_id']


,activity_id,drug_chembl_id,drug_name,molecule_type,molecular_weight,canonical_smiles,target_chembl_id,target_name,organism,standard_type,...,assay_type,assay_description,assay_organism,confidence_score,article_title,journal,year,pubmed_id,doi,uniprot_id
0,12645440,CHEMBL2322194,None,Small molecule,445.55,NS(=O)(=O)OC[C@@H]1C[C@@H](N2CCc3c(N[C@H]4CCc5...,CHEMBL2321622,Ubiquitin-like modifier-activating enzyme 6,Homo sapiens,IC50,...,B,Inhibition of UBA6 (unknown origin),Homo sapiens,9,Exploring a new frontier in cancer treatment: ...,J Med Chem,2013.0,23360215.0,10.1021/jm301420b,A0AVT1
1,12645445,CHEMBL2017005,None,Small molecule,462.49,NS(=O)(=O)OC[C@H]1O[C@@H](n2cnc3c(N[C@H]4CCc5c...,CHEMBL2321622,Ubiquitin-like modifier-activating enzyme 6,Homo sapiens,IC50,...,B,Inhibition of UBA6 (unknown origin) in presenc...,Homo sapiens,9,Exploring a new frontier in cancer treatment: ...,J Med Chem,2013.0,23360215.0,10.1021/jm301420b,A0AVT1
2,18483955,CHEMBL1231160,PEVONEDISTAT,Small molecule,443.53,NS(=O)(=O)OC[C@@H]1C[C@@H](n2ccc3c(N[C@H]4CCc5...,CHEMBL2321622,Ubiquitin-like modifier-activating enzyme 6,Homo sapiens,IC50,...,B,Inhibition of UBA6 (unknown origin) assessed a...,Homo sapiens,9,Interrogating the Roles of Post-Translational ...,J Med Chem,2018.0,28505447.0,10.1021/acs.jmedchem.6b01817,A0AVT1



────────────────────────────────────────────────────────────
  PDBe  —  columns: ['uniprot_id', 'pdb_id', 'chain_id', 'resolution', 'coverage', 'unp_start', 'unp_end', 'method']


,uniprot_id,pdb_id,chain_id,resolution,coverage,unp_start,unp_end,method
0,A0AVT1,7pvn,A,2.71,1.0,1,1052,X-ray diffraction
1,A0AVT1,7pvn,B,2.71,1.0,1,1052,X-ray diffraction
2,A0AVT1,9qh5,B,3.09,1.0,1,1052,Electron Microscopy



────────────────────────────────────────────────────────────
  PubMed  —  columns: ['pubmed_id', 'abstract', 'authors', 'pub_date', 'year', 'month', 'doi']


,pubmed_id,abstract,authors,pub_date,year,month,doi
0,23360215,The labeling of proteins with small ubiquitin ...,"da Silva, Sara R; Paiva, Stacey-Lynn; Lukkaril...",2013-3-28,2013,3.0,10.1021/jm301420b
1,28505447,Post-translational modifications (PTMs) allot ...,"Buuh, Zakey Yusuf; Lyu, Zhigang; Wang, Rongshe...",2018-4-26,2018,4.0,10.1021/acs.jmedchem.6b01817
2,29501416,The human O-acetyl-ADP-ribose deacetylase MDO1...,"Zhang, Yuezhou; Jumppanen, Mikael; Maksimainen...",2018-5-01,2018,5.0,10.1016/j.bmc.2018.02.006


## 3 · Clean & Normalise Proteins — `DimProtein`

Steps applied by `DataCleaner.clean_uniprot()`:
- Assert that the required columns (`Entry`, `Gene Names`, `Protein names`) are present.
- Rename to the schema names: `accession`, `gene_names`, `protein_name`.
- Uppercase-normalise `accession`.
- Drop rows where `accession` is null.
- Deduplicate on `accession`.


In [4]:
cleaner = DataCleaner()

if not df_raw_uniprot.empty:
    df_clean_uniprot = cleaner.clean_uniprot(df_raw_uniprot)
else:
    # Fallback: rename columns that the real file would have produced
    print("⚠  UniProt CSV missing — using empty placeholder.")
    df_clean_uniprot = pd.DataFrame(columns=["accession", "gene_names", "protein_name"])

print(f"\nDimProtein source shape  : {df_clean_uniprot.shape}")
print(f"Null accessions          : {df_clean_uniprot['accession'].isna().sum()}")
print(f"Unique accessions        : {df_clean_uniprot['accession'].nunique()}")
display(df_clean_uniprot.head())


2026-03-26 05:23:47  INFO      src.transformation.cleaner  UniProt cleaning complete – 20431 proteins retained.

DimProtein source shape  : (20431, 3)
Null accessions          : 0
Unique accessions        : 20431

DimProtein source shape  : (20431, 3)
Null accessions          : 0
Unique accessions        : 20431


,accession,gene_names,protein_name
0,A0A087X1C5,CYP2D7,Cytochrome P450 2D7 (EC 1.14.14.1)
1,A0A096LP01,SMIM26 LINC00493,Small integral membrane protein 26
2,A0A0B4J2F0,PIGBOS1,Protein PIGBOS1 (PIGB opposite strand protein 1)
3,A0A0C5B5G6,MT-RNR1,Mitochondrial-derived peptide MOTS-c (Mitochon...
4,A0A0K2S4Q6,CD300H,Protein CD300H (CD300 antigen-like family memb...


## 4 · Clean & Normalise Drugs — `DimDrug`

Steps applied by `DataCleaner.clean_chembl()`:
- Rename columns: `drug_chembl_id`, `drug_name`, `uniprot_id`, `standard_type`, `standard_value`, etc.
- Normalise `uniprot_id` (strip non-alphanumeric characters, uppercase).
- Cast `standard_value` and `pchembl_value` to `Float64`.
- Cast `confidence_score` and `year` to `Int64`.
- Drop rows where both `drug_chembl_id` and `uniprot_id` are null.
- Deduplicate on `activity_id`.

`DimDrug` is then extracted as the unique drug sub-set of the cleaned activities.


In [6]:
if not df_raw_chembl.empty:
    df_clean_activities = cleaner.clean_chembl(df_raw_chembl)
else:
    print("⚠  Activities CSV missing — using empty placeholder.")
    df_clean_activities = pd.DataFrame(columns=[
        "activity_id", "drug_chembl_id", "drug_name", "uniprot_id",
        "standard_type", "standard_value", "standard_units", "pchembl_value",
        "assay_type", "assay_description", "confidence_score",
        "article_title", "journal", "year", "pubmed_id",
    ])

# Extract unique drugs for DimDrug
DIM_DRUG_COLS = ["drug_chembl_id", "drug_name"]
df_dim_drug_source = (
    df_clean_activities[DIM_DRUG_COLS]
    .dropna(subset=["drug_chembl_id"])
    .drop_duplicates(subset=["drug_chembl_id"])
    .reset_index(drop=True)
)

print(f"\nClean activities shape   : {df_clean_activities.shape}")
print(f"Unique drugs (DimDrug)   : {len(df_dim_drug_source)}")
display(df_dim_drug_source.head())


2026-03-26 05:25:49  INFO      src.transformation.cleaner  ChEMBL cleaning complete – 6441095 bioactivity records retained.

Clean activities shape   : (6441095, 23)
Unique drugs (DimDrug)   : 1509932

Clean activities shape   : (6441095, 23)
Unique drugs (DimDrug)   : 1509932


,drug_chembl_id,drug_name
0,CHEMBL2322194,None
1,CHEMBL2017005,None
2,CHEMBL1231160,PEVONEDISTAT
3,CHEMBL4226903,None
4,CHEMBL5205107,None


## 5 · Clean & Normalise Articles — `DimArticle`

`DimArticle` is assembled by:
1. Extracting unique bibliographic records (`pubmed_id`, `article_title`, `journal`, `year`) from the cleaned activities.
2. Merging with `article_abstracts.csv` on `pubmed_id` to add the `abstract_text` column.
3. Applying `DataCleaner.clean_pubmed()` to collapse whitespace and remove null abstracts.

`doi` and `first_author` are included when available in the raw data.


In [7]:
# ── Step 1: unique bibliography from activities ───────────────────────────────
ARTICLE_META_COLS = ["pubmed_id", "article_title", "journal", "year"]
available_meta = [c for c in ARTICLE_META_COLS if c in df_clean_activities.columns]

df_article_meta = (
    df_clean_activities[available_meta]
    .dropna(subset=["pubmed_id"])
    .drop_duplicates(subset=["pubmed_id"])
    .reset_index(drop=True)
)

# ── Step 2: merge abstracts ───────────────────────────────────────────────────
if not df_raw_pubmed.empty:
    df_clean_abstracts = cleaner.clean_pubmed(df_raw_pubmed)
    df_dim_article_source = df_article_meta.merge(
        df_clean_abstracts[["pubmed_id", "abstract"]].rename(columns={"abstract": "abstract_text"}),
        on="pubmed_id",
        how="left",
    )
else:
    df_dim_article_source = df_article_meta.copy()
    df_dim_article_source["abstract_text"] = pd.NA

# ── Step 3: add optional columns ─────────────────────────────────────────────
for optional_col in ("doi", "first_author"):
    if optional_col not in df_dim_article_source.columns:
        df_dim_article_source[optional_col] = pd.NA

print(f"\nDimArticle shape          : {df_dim_article_source.shape}")
print(f"Articles with abstracts   : {df_dim_article_source['abstract_text'].notna().sum()}")
display(df_dim_article_source.head())


2026-03-26 05:26:41  INFO      src.transformation.cleaner  PubMed cleaning complete – 35551 abstracts retained.

DimArticle shape          : (36351, 7)
Articles with abstracts   : 34775

DimArticle shape          : (36351, 7)
Articles with abstracts   : 34775


,pubmed_id,article_title,journal,year,abstract_text,doi,first_author
0,23360215,Exploring a new frontier in cancer treatment: ...,J Med Chem,2013,The labeling of proteins with small ubiquitin ...,<NA>,<NA>
1,28505447,Interrogating the Roles of Post-Translational ...,J Med Chem,2018,Post-translational modifications (PTMs) allot ...,<NA>,<NA>
2,29501416,Adenosine analogs bearing phosphate isosteres ...,Bioorg Med Chem,2018,The human O-acetyl-ADP-ribose deacetylase MDO1...,<NA>,<NA>
3,35131538,NAE modulators: A potential therapy for gastri...,Eur J Med Chem,2022,Neural precursor cell expressed developmentall...,<NA>,<NA>
4,35597097,"Design, synthesis and evaluation of inhibitors...",Bioorg Med Chem,2022,"A series of amino acid based 7H-pyrrolo[2,3-d]...",<NA>,<NA>


## 6 · Clean & Normalise Structures — `DimStructure`

Steps applied by `DataCleaner.clean_pdbe()`:
- Assert `pdb_id`, `uniprot_id`, `chain_id` are present.
- Drop rows where `pdb_id` is null.
- Cast `resolution` and `coverage` to `Float64`.
- Cast `unp_start` / `unp_end` to `Int64`.
- Normalise `uniprot_id` (strip non-alphanumeric, uppercase).
- Deduplicate on (`pdb_id`, `chain_id`).

> **Note:** `resolution` is measured in **Angstroms** ($10^{-10}$ m). Lower values indicate higher-quality structures.


In [8]:
if not df_raw_pdbe.empty:
    df_clean_pdb = cleaner.clean_pdbe(df_raw_pdbe)
else:
    print("⚠  PDB CSV missing — using empty placeholder.")
    df_clean_pdb = pd.DataFrame(columns=[
        "pdb_id", "chain_id", "uniprot_id",
        "resolution", "coverage", "method", "unp_start", "unp_end",
    ])

print(f"\nDimStructure shape        : {df_clean_pdb.shape}")
print(f"Unique PDB IDs            : {df_clean_pdb['pdb_id'].nunique()}")
print("\nResolution statistics (Å) :")
display(df_clean_pdb["resolution"].describe().to_frame().T)

display(df_clean_pdb.head())


2026-03-26 05:27:28  INFO      src.transformation.cleaner  PDBe cleaning complete – 165977 structure entries retained.

DimStructure shape        : (165977, 8)
Unique PDB IDs            : 61839

Resolution statistics (Å) :

DimStructure shape        : (165977, 8)
Unique PDB IDs            : 61839

Resolution statistics (Å) :


,count,mean,std,min,25%,50%,75%,max
resolution,162400.0,2.756262,1.668277,0.6,2.0,2.55,3.12,53.0


,uniprot_id,pdb_id,chain_id,resolution,coverage,unp_start,unp_end,method
0,A0AVT1,7PVN,A,2.71,1.0,1,1052,X-ray diffraction
1,A0AVT1,7PVN,B,2.71,1.0,1,1052,X-ray diffraction
2,A0AVT1,9QH5,B,3.09,1.0,1,1052,Electron Microscopy
3,A0AVT1,9QIC,B,3.29,1.0,1,1052,Electron Microscopy
4,A0AVT1,9QIV,B,3.44,1.0,1,1052,Electron Microscopy


## 7 · Build `FactBioactivity`

The fact table is derived directly from the cleaned activities DataFrame.

| Column | Source | Notes |
|---|---|---|
| `uniprot_id_fk` | `uniprot_id` | FK → `DimProtein` |
| `drug_id_fk` | `drug_chembl_id` | FK → `DimDrug` |
| `pubmed_id_fk` | `pubmed_id` | FK → `DimArticle` (nullable) |
| `standard_type` | `standard_type` | e.g. IC50, Ki, Kd |
| `standard_value` | `standard_value` | Numerical potency |
| `standard_units` | `standard_units` | e.g. nM, µM |
| `pchembl_value` | `pchembl_value` | −log₁₀(molar potency) |
| `confidence_score` | `confidence_score` | ChEMBL 0–9 rating |
| `assay_type` | `assay_type` | B / F / A … |
| `assay_description` | `assay_description` | Free-text assay context |

Rows with a null `standard_value` are dropped because they carry no quantitative evidence.


In [9]:
FACT_RENAME = {
    "uniprot_id":        "uniprot_id_fk",
    "drug_chembl_id":    "drug_id_fk",
    "pubmed_id":         "pubmed_id_fk",
}

FACT_COLS = [
    "uniprot_id_fk", "drug_id_fk", "pubmed_id_fk",
    "standard_type", "standard_value", "standard_units",
    "pchembl_value", "confidence_score",
    "assay_type", "assay_description",
]

if not df_clean_activities.empty:
    df_fact = (
        df_clean_activities
        .rename(columns=FACT_RENAME)
        .dropna(subset=["standard_value"])          # require a numeric measurement
        .reindex(columns=FACT_COLS)                 # keep only fact columns
        .reset_index(drop=True)
    )
else:
    df_fact = pd.DataFrame(columns=FACT_COLS)

print(f"FactBioactivity shape     : {df_fact.shape}")
print(f"Null pchembl_value        : {df_fact['pchembl_value'].isna().sum()}")
display(df_fact.head())


FactBioactivity shape     : (6441095, 10)
Null pchembl_value        : 3872316


,uniprot_id_fk,drug_id_fk,pubmed_id_fk,standard_type,standard_value,standard_units,pchembl_value,confidence_score,assay_type,assay_description
0,A0AVT1,CHEMBL2322194,23360215,IC50,1800.0,nM,5.75,9,B,Inhibition of UBA6 (unknown origin)
1,A0AVT1,CHEMBL2017005,23360215,IC50,920.0,nM,6.04,9,B,Inhibition of UBA6 (unknown origin) in presenc...
2,A0AVT1,CHEMBL1231160,28505447,IC50,1000.0,nM,NaN,9,B,Inhibition of UBA6 (unknown origin) assessed a...
3,A1Z1Q3,CHEMBL4226903,29501416,Kd,150.0,nM,6.82,9,B,Binding affinity to human MDO2 by ITC
4,A0AVT1,CHEMBL1231160,35131538,IC50,1800.0,nM,5.75,9,B,Inhibition of His-tagged UBA6 (unknown origin)...


**NOTE:** assay_type = B is not very readable, maybe check again the db schema for metadata or select a different column to capture the assay type in a more interpretable way.

## 8 · Persist Staging Files & Initialise Warehouse Schema

### 8a — Staging Parquet files
Before touching the database we persist the cleaned dimension and fact DataFrames to `data/staging/` as **Parquet**.  
This gives us a reproducible intermediate layer: if the DB load fails, we can reload from staging without re-running extraction.

### 8b — Warehouse schema
We use **SQLite** for the demo warehouse so the notebook runs without a MySQL server.  
The DDL is read directly from `src/database/schema.sql` — the same file you would run against MySQL in production:
```bash
mysql -u <user> -p eliqsir_dw < src/database/schema.sql
```
SQLite does not support all MySQL DDL syntax, so MySQL-specific clauses (`ENGINE`, `CHARSET`, `FULLTEXT`, `TINYINT UNSIGNED`, `DOUBLE`) are stripped automatically before execution.


In [10]:
# ── 8a: Save staging Parquet ─────────────────────────────────────────────────
staging_frames = {
    "dim_protein":      df_clean_uniprot,
    "dim_drug":         df_dim_drug_source,
    "dim_article":      df_dim_article_source,
    "dim_structure":    df_clean_pdb,
    "fact_bioactivity": df_fact,
}

for name, df in staging_frames.items():
    out = STAGING_DIR / f"{name}.parquet"
    df.to_parquet(out, index=False)
    print(f"  ✓  {name:<22} {len(df):>8,} rows  →  {out.name}")

print(f"\nAll staging files written to: {_dp(STAGING_DIR)}")

# ── 8b: Initialise SQLite warehouse from schema.sql ──────────────────────────
def _mysql_to_sqlite_ddl(sql: str) -> str:
    """Strip MySQL-specific clauses so the DDL runs on SQLite."""
    # Remove ENGINE / CHARSET / COLLATE / COMMENT table options
    sql = re.sub(r"ENGINE\s*=\s*\w+", "", sql, flags=re.IGNORECASE)
    sql = re.sub(r"DEFAULT\s+CHARSET\s*=\s*\w+", "", sql, flags=re.IGNORECASE)
    sql = re.sub(r"COLLATE\s*=?\s*\w+", "", sql, flags=re.IGNORECASE)
    sql = re.sub(r"COMMENT\s*=\s*'[^']*'", "", sql, flags=re.IGNORECASE)
    # Remove inline column COMMENT '...'
    sql = re.sub(r"COMMENT\s+'[^']*'", "", sql, flags=re.IGNORECASE)
    # FULLTEXT KEY → INDEX (SQLite doesn't support it)
    sql = re.sub(r"FULLTEXT\s+KEY\s+\w+\s*\([^)]+\),?\s*", "", sql, flags=re.IGNORECASE)
    # TINYINT UNSIGNED → INTEGER
    sql = re.sub(r"TINYINT\s+UNSIGNED", "INTEGER", sql, flags=re.IGNORECASE)
    # DOUBLE → REAL
    sql = re.sub(r"\bDOUBLE\b", "REAL", sql, flags=re.IGNORECASE)
    # SMALLINT → INTEGER
    sql = re.sub(r"\bSMALLINT\b", "INTEGER", sql, flags=re.IGNORECASE)
    # BIGINT → INTEGER
    sql = re.sub(r"\bBIGINT\b", "INTEGER", sql, flags=re.IGNORECASE)
    # VARCHAR(n) → TEXT
    sql = re.sub(r"VARCHAR\(\d+\)", "TEXT", sql, flags=re.IGNORECASE)
    # Remove trailing commas before closing paren (left by stripped FULLTEXT lines)
    sql = re.sub(r",\s*\)", "\n)", sql)
    return sql


WAREHOUSE_DB.parent.mkdir(parents=True, exist_ok=True)
conn_wh = sqlite3.connect(WAREHOUSE_DB)

raw_ddl   = SCHEMA_SQL.read_text(encoding="utf-8")
sqlite_ddl = _mysql_to_sqlite_ddl(raw_ddl)

cursor_wh = conn_wh.cursor()
cursor_wh.executescript(sqlite_ddl)   # executescript handles multiple statements
conn_wh.commit()

cursor_wh.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
tables = [r[0] for r in cursor_wh.fetchall()]
print("\nTables in warehouse:")
for t in tables:
    print(f"  ✓  {t}")


  ✓  dim_protein              20,431 rows  →  dim_protein.parquet
  ✓  dim_drug               1,509,932 rows  →  dim_drug.parquet
  ✓  dim_drug               1,509,932 rows  →  dim_drug.parquet
  ✓  dim_article              36,351 rows  →  dim_article.parquet
  ✓  dim_structure           165,977 rows  →  dim_structure.parquet
  ✓  dim_article              36,351 rows  →  dim_article.parquet
  ✓  dim_structure           165,977 rows  →  dim_structure.parquet
  ✓  fact_bioactivity       6,441,095 rows  →  fact_bioactivity.parquet

All staging files written to: data/staging
  ✓  fact_bioactivity       6,441,095 rows  →  fact_bioactivity.parquet

All staging files written to: data/staging


OperationalError: near "SET": syntax error

## 9 · Load Dimensions from Staging into the Warehouse

We read the dimension Parquet files from `data/staging/` and insert them using `INSERT OR IGNORE` (SQLite equivalent of MySQL's `ON DUPLICATE KEY IGNORE`).  
Loading **from staging files** rather than in-memory DataFrames means the load step can be re-run independently without re-executing the full transformation.


In [11]:
def count_rows(conn: sqlite3.Connection, table: str) -> int:
    return conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]


def load_dim(
    conn: sqlite3.Connection,
    df: pd.DataFrame,
    table: str,
    columns: list[str],
    *,
    natural_key: str,
) -> None:
    """Insert rows from *df* into *table* using INSERT OR IGNORE (idempotent)."""
    before = count_rows(conn, table)

    rows_to_insert = [
        tuple(row[c] if c in row and pd.notna(row[c]) else None for c in columns)
        for _, row in df.iterrows()
        if pd.notna(row.get(natural_key))
    ]

    if rows_to_insert:
        placeholders = ", ".join(["?"] * len(columns))
        sql = (
            f"INSERT OR IGNORE INTO {table} "
            f"({', '.join(columns)}) VALUES ({placeholders})"
        )
        conn.executemany(sql, rows_to_insert)
        conn.commit()

    after = count_rows(conn, table)
    print(f"  {table:<22}  before={before:>7,}  after={after:>7,}  inserted={after - before:>7,}")


print("Loading dimensions from staging Parquet …")

# Re-read from staging to demonstrate the staging-→-DB pattern
load_dim(
    conn_wh,
    pd.read_parquet(STAGING_DIR / "dim_protein.parquet"),
    "dim_protein",
    ["uniprot_id", "protein_name", "gene_names"],
    natural_key="uniprot_id",
)
load_dim(
    conn_wh,
    pd.read_parquet(STAGING_DIR / "dim_drug.parquet"),
    "dim_drug",
    ["drug_chembl_id", "drug_name"],
    natural_key="drug_chembl_id",
)
load_dim(
    conn_wh,
    pd.read_parquet(STAGING_DIR / "dim_article.parquet"),
    "dim_article",
    ["pubmed_id", "article_title", "journal", "year", "abstract_text", "doi", "first_author"],
    natural_key="pubmed_id",
)
load_dim(
    conn_wh,
    pd.read_parquet(STAGING_DIR / "dim_structure.parquet"),
    "dim_structure",
    ["pdb_id", "chain_id", "uniprot_id", "resolution", "coverage", "method", "unp_start", "unp_end"],
    natural_key="pdb_id",
)

print("\n✓  All dimensions loaded.")


Loading dimensions from staging Parquet …


OperationalError: no such table: dim_protein

## 10 · Load Fact Table into the Warehouse

Before inserting, we perform a **referential integrity pre-check**:
- All `uniprot_id_fk` values must exist in `dim_protein`.
- All `drug_id_fk` values must exist in `dim_drug`.
- `pubmed_id_fk` is nullable but, when present, must match a `dim_article` row.

Rows that fail the FK check are logged and excluded from insertion.


In [ ]:
def fetch_set(conn: sqlite3.Connection, table: str, column: str) -> set:
    rows = conn.execute(f"SELECT DISTINCT {column} FROM {table}").fetchall()
    return {r[0] for r in rows}


# ── Load fact from staging Parquet ────────────────────────────────────────────
df_fact_load = pd.read_parquet(STAGING_DIR / "fact_bioactivity.parquet")
print(f"Loaded from staging: {len(df_fact_load):,} fact rows")

# ── FK sets from the already-loaded dimension tables ─────────────────────────
valid_proteins = fetch_set(conn_wh, "dim_protein", "uniprot_id")
valid_drugs    = fetch_set(conn_wh, "dim_drug",    "drug_chembl_id")
valid_articles = fetch_set(conn_wh, "dim_article", "pubmed_id")

# ── referential integrity pre-check ──────────────────────────────────────────
mask_protein = df_fact_load["uniprot_id_fk"].isin(valid_proteins)
mask_drug    = df_fact_load["drug_id_fk"].isin(valid_drugs)
mask_article = (
    df_fact_load["pubmed_id_fk"].isna() |
    df_fact_load["pubmed_id_fk"].isin(valid_articles)
)

orphans = ~(mask_protein & mask_drug & mask_article)
print(f"Rows with FK violations   : {orphans.sum():,}")

df_fact_clean = df_fact_load[~orphans].reset_index(drop=True)
print(f"Rows eligible for load    : {len(df_fact_clean):,}")

# ── insert ────────────────────────────────────────────────────────────────────
FACT_COLS_DB = [
    "uniprot_id_fk", "drug_id_fk", "pubmed_id_fk",
    "standard_type", "standard_value", "standard_units",
    "pchembl_value", "confidence_score", "assay_type", "assay_description",
]

before_fact = count_rows(conn_wh, "fact_bioactivity")

if not df_fact_clean.empty:
    rows = [
        tuple(None if pd.isna(row[c]) else row[c] for c in FACT_COLS_DB)
        for _, row in df_fact_clean.iterrows()
    ]
    ph = ", ".join(["?"] * len(FACT_COLS_DB))
    conn_wh.executemany(
        f"INSERT INTO fact_bioactivity ({', '.join(FACT_COLS_DB)}) VALUES ({ph})",
        rows,
    )
    conn_wh.commit()

after_fact = count_rows(conn_wh, "fact_bioactivity")
print(f"\n✓  fact_bioactivity: inserted {after_fact - before_fact:,} rows  (total={after_fact:,})")


## 11 · Validate Loaded Data

Run a set of SQL validation queries to confirm:
1. **Row counts** for all five tables.
2. **Orphan FK check** — fact rows whose `uniprot_id_fk` or `drug_id_fk` has no matching dimension row.
3. **Non-null constraint** — no nulls in critical dimension key columns.
4. **Coverage** — percentage of fact rows that have a linked abstract.


In [ ]:
# ── 1. Row counts ─────────────────────────────────────────────────────────────
validation_queries = {
    "dim_protein row count":    "SELECT COUNT(*) FROM dim_protein",
    "dim_drug row count":       "SELECT COUNT(*) FROM dim_drug",
    "dim_article row count":    "SELECT COUNT(*) FROM dim_article",
    "dim_structure row count":  "SELECT COUNT(*) FROM dim_structure",
    "fact_bioactivity row count": "SELECT COUNT(*) FROM fact_bioactivity",
    # 2. Orphan FKs
    "orphan uniprot_id_fk": """
        SELECT COUNT(*) FROM fact_bioactivity f
        WHERE f.uniprot_id_fk NOT IN (SELECT uniprot_id FROM dim_protein)
    """,
    "orphan drug_id_fk": """
        SELECT COUNT(*) FROM fact_bioactivity f
        WHERE f.drug_id_fk NOT IN (SELECT drug_chembl_id FROM dim_drug)
    """,
    # 3. Null key columns
    "dim_protein null accession": "SELECT COUNT(*) FROM dim_protein WHERE uniprot_id IS NULL",
    "dim_drug null chembl_id":    "SELECT COUNT(*) FROM dim_drug WHERE drug_chembl_id IS NULL",
    # 4. Abstract coverage
    "fact rows with abstract": """
        SELECT COUNT(*) FROM fact_bioactivity f
        WHERE f.pubmed_id_fk IN (SELECT pubmed_id FROM dim_article WHERE abstract_text IS NOT NULL)
    """,
}

results = []
for check, sql in validation_queries.items():
    value = conn_wh.execute(sql).fetchone()[0]
    status = "✓" if (
        ("orphan" in check and value == 0) or
        ("null" in check and value == 0) or
        ("count" in check) or
        ("with abstract" in check)
    ) else "⚠"
    results.append({"check": check, "value": value, "status": status})

df_validation = pd.DataFrame(results)
display(df_validation.style.set_caption("Warehouse Validation Report"))

print("\n✓  Validation complete.")
conn_wh.close()
